# H₂ Binding Energy via VQE

Compute the ground-state energy of molecular hydrogen using the Variational Quantum Eigensolver, then derive the binding energy $E_b = 2\,E(\mathrm{H}) - E(\mathrm{H}_2)$.

**Why H₂ and not WH?** Tungsten chemistry is a real research target (tritium retention in fusion reactor first walls), but it lives at 100+ qubits with sophisticated effective-core-potential basis sets. Even the small-active-space version requires `pyscf`, which doesn't build on Windows. H₂ is the canonical 'works everywhere' starting point — 4-Pauli Hamiltonian after parity-mapping with qubit reduction, runs locally in seconds, matches Full Configuration Interaction to microhartrees.

Once this notebook makes sense end-to-end, the natural progression is LiH (12 qubits), then BeH₂, then transition metals in WSL with `pyscf`.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import efficient_su2
from qiskit.primitives import StatevectorEstimator

HARTREE_TO_EV = 27.211386245988

## 1. The H₂ Hamiltonian

These coefficients are the STO-3G basis H₂ electronic Hamiltonian at bond length **r = 0.735 Å** (near equilibrium), mapped to qubits with the parity transformation and reduced by 2-qubit tapering. Source: O'Malley et al., *Scalable Quantum Simulation of Molecular Energies*, Phys. Rev. X **6**, 031007 (2016), Appendix B — the same numbers used in dozens of subsequent VQE benchmarks.

Only 5 Pauli terms on 2 qubits. The exact ground state energy is known to be **–1.857 Ha** (electronic energy; add the constant nuclear repulsion ~0.7137 Ha to get the total).

In [ ]:
H2 = SparsePauliOp.from_list([
    ('II', -1.052373245772859),
    ('IZ',  0.39793742484318045),
    ('ZI', -0.39793742484318045),
    ('ZZ', -0.01128010425623538),
    ('XX',  0.18093119978423156),
])
print(H2)
print('qubits:', H2.num_qubits)
print('Pauli terms:', len(H2))

## 2. Exact reference

At 2 qubits the Hamiltonian is a 4×4 matrix — diagonalizable classically in microseconds. We compute the exact ground state energy first, then VQE has a target to hit. (At >40 qubits this becomes infeasible classically, which is why we need quantum hardware.)

In [ ]:
eigenvalues = np.linalg.eigvalsh(H2.to_matrix())
E_exact = eigenvalues[0]
print(f'Exact ground state energy: {E_exact:.10f} Ha')
print(f'                          = {E_exact * HARTREE_TO_EV:.5f} eV')

## 3. Variational ansatz

`efficient_su2` is a hardware-efficient ansatz: alternating layers of single-qubit rotations (around Y and Z axes) interleaved with entangling CX gates. Two reps gives 8 parameters — plenty for a 2-qubit problem.

(This is the 'hardware-friendly' style, not the chemistry-aware UCCSD style. For H₂ either works; for larger molecules UCCSD has theoretical advantages but is harder to implement on noisy hardware.)

In [ ]:
ansatz = efficient_su2(num_qubits=2, reps=2, entanglement='linear')
print('parameters:', ansatz.num_parameters)
ansatz.decompose().draw('mpl')

## 4. The VQE loop

Classical optimizer (COBYLA) drives quantum-evaluated expectation values. Each call to `cost(params)` is, in spirit, one quantum experiment: prepare $|\psi(\theta)\rangle$ on hardware, measure the Hamiltonian, get the expectation value, return it to the optimizer.

Here we use `StatevectorEstimator` which is the noiseless local simulator — fast and deterministic, the right starting point. Swapping in `BackendEstimatorV2(backend)` would send each evaluation to a real QPU (and absorb its queue time).

In [ ]:
estimator = StatevectorEstimator()
history = []

def cost(params):
    job = estimator.run([(ansatz, H2, params)])
    energy = float(job.result()[0].data.evs)
    history.append(energy)
    return energy

rng = np.random.default_rng(seed=7)
x0 = rng.uniform(0, 2 * np.pi, ansatz.num_parameters)

result = minimize(cost, x0=x0, method='COBYLA', options={'maxiter': 300, 'rhobeg': 0.5})
E_vqe = result.fun

print(f'VQE energy            : {E_vqe:.10f} Ha')
print(f'Exact reference       : {E_exact:.10f} Ha')
print(f'Error                 : {abs(E_vqe - E_exact):.2e} Ha  ({abs(E_vqe-E_exact)*HARTREE_TO_EV*1000:.3f} meV)')
print(f'Cost-function calls   : {len(history)}')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(history, color='#7F77DD', lw=1)
plt.axhline(E_exact, ls='--', color='black', alpha=0.6, label=f'exact: {E_exact:.5f} Ha')
plt.xlabel('cost function call')
plt.ylabel('energy (Ha)')
plt.title('VQE optimization trajectory — H\u2082 ground state')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Binding energy

Binding energy compares the bound molecule against its separated atoms:
$$E_b = 2\,E(\mathrm{H}) - E(\mathrm{H}_2)$$

$E(\mathrm{H}_2)$ above is the **electronic** energy. To get the total energy we add the nuclear repulsion $V_{NN}(r) = 1/r$ in atomic units. At r = 0.735 Å = 1.3886 Bohr, $V_{NN}$ = 0.7201 Ha.

For a single H atom in STO-3G, the energy is $E(\mathrm{H})_{STO\text{-}3G}$ = –0.4666 Ha (compared to the exact –0.5 Ha — STO-3G is a crude basis, the limitation isn't the quantum algorithm).

**STO-3G is known to overbind covalent bonds**, so don't be surprised if our number exceeds the experimental 4.52 eV.

In [ ]:
BOHR_PER_ANGSTROM = 1.8897259886
r_angstrom = 0.735
r_bohr = r_angstrom * BOHR_PER_ANGSTROM
V_NN = 1.0 / r_bohr  # nuclear repulsion in atomic units

E_H_atom_STO3G = -0.4665818  # well-known STO-3G energy of a single H atom

E_H2_total = E_vqe + V_NN
E_separated = 2 * E_H_atom_STO3G
E_binding = E_separated - E_H2_total

print(f'r              = {r_angstrom} Å  ({r_bohr:.4f} Bohr)')
print(f'V_NN(r)        = {V_NN:.6f} Ha')
print(f'E(H\u2082) total   = {E_H2_total:.6f} Ha   (VQE electronic + V_NN)')
print(f'2 × E(H atom)  = {E_separated:.6f} Ha   (STO-3G reference)')
print()
print(f'Binding energy = {E_binding:.6f} Ha')
print(f'               = {E_binding * HARTREE_TO_EV:.4f} eV')
print()
print(f'Experimental D_e(H\u2082) = 4.52 eV')
print(f'STO-3G overbinds by {E_binding * HARTREE_TO_EV - 4.52:+.2f} eV — '
      f'expected; this is a basis-set limitation, not a quantum-algorithm limitation.')

## Next steps

- **Better basis**: redo the same H₂ problem with a 6-31G or cc-pVDZ   Hamiltonian — the qubit count grows but the binding energy   converges toward the experimental value.
- **Real QPU**: replace `StatevectorEstimator()` with   `BackendEstimatorV2(backend)` from `qiskit_ibm_runtime` to run each   cost evaluation on real hardware. Expect significant noise; you'll   need shot averaging and error mitigation.
- **Larger molecules**: LiH (12 qubits) is the next logical step —   still classically tractable, but a meaningful chemistry problem.
- **Tungsten chemistry**: requires `pyscf` + ECP basis sets, which   means setting up WSL2 with a Linux Python environment. The W-vacancy/H   binding problem also needs 50–100+ qubits of active space —   beyond current statevector simulators, demands real hardware with   error correction.